In [1]:
import wfdb

filer = [
    "ptbxl_database.csv",
    "scp_statements.csv",
    "records100/00000/00001_lr.hea",
    "records100/00000/00001_lr.dat",
    "records100/00000/00002_lr.hea",
    "records100/00000/00002_lr.dat",
    "records100/00000/00003_lr.hea",
    "records100/00000/00003_lr.dat",
    "records500/00000/00001_hr.hea",
    "records500/00000/00001_hr.dat",
    "records500/00000/00002_hr.hea",
    "records500/00000/00002_hr.dat",
    "records500/00000/00003_hr.hea",
    "records500/00000/00003_hr.dat",
]

wfdb.dl_files("ptb-xl", "data/ptbxl_mini", filer)


Finished downloading files


In [2]:
import pandas as pd

df = pd.read_csv("data/ptbxl_mini/ptbxl_database.csv", index_col="ecg_id")

print(df.shape)

(21799, 27)


In [3]:
df.head(
    
)

,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
ecg_id,,,,,,,,,,,,,,,,,,,,,
1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,sinusrhythmus periphere niederspannung,...,True,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr
2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,sinusbradykardie sonst normales ekg,...,True,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr
3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,sinusrhythmus normales ekg,...,True,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr
4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,sinusrhythmus normales ekg,...,True,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr
5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,sinusrhythmus normales ekg,...,True,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr


In [4]:
import ast

df["scp_codes"] = df["scp_codes"].apply(ast.literal_eval)

In [5]:
df["scp_codes"].head(10)


ecg_id
1     {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
2                 {'NORM': 80.0, 'SBRAD': 0.0}
3                   {'NORM': 100.0, 'SR': 0.0}
4                   {'NORM': 100.0, 'SR': 0.0}
5                   {'NORM': 100.0, 'SR': 0.0}
6                   {'NORM': 100.0, 'SR': 0.0}
7                   {'NORM': 100.0, 'SR': 0.0}
8       {'IMI': 35.0, 'ABQRS': 0.0, 'SR': 0.0}
9                   {'NORM': 100.0, 'SR': 0.0}
10                  {'NORM': 100.0, 'SR': 0.0}
Name: scp_codes, dtype: object

In [6]:
ordbok = pd.read_csv("data/ptbxl_mini/scp_statements.csv", index_col=0)

diagnoser = ordbok[ordbok["diagnostic"] == 1]

print(len(ordbok))
print(len(diagnoser))

71
44


In [7]:
koder = df.loc[8, "scp_codes"]
print(koder)

for kode, sikkerhet in koder.items():
    print(kode, sikkerhet)
    

{'IMI': 35.0, 'ABQRS': 0.0, 'SR': 0.0}
IMI 35.0
ABQRS 0.0
SR 0.0


In [9]:
koder = df.loc[8, "scp_codes"]
for kode in koder:
    print(kode, kode in diagnoser.index)

IMI True
ABQRS False
SR False


In [10]:
def finn_diagnoser(koder):
    funnet = []
    for kode in koder:
        if kode in diagnoser.index:
            funnet.append(kode)
    return funnet

In [11]:
print(finn_diagnoser(df.loc[8, "scp_codes"]))
print(finn_diagnoser(df.loc[1, "scp_codes"]))

['IMI']
['NORM']


In [12]:
df["diagnoser"] = df["scp_codes"].apply(finn_diagnoser)
df["diagnoser"].head(10)

ecg_id
1     [NORM]
2     [NORM]
3     [NORM]
4     [NORM]
5     [NORM]
6     [NORM]
7     [NORM]
8      [IMI]
9     [NORM]
10    [NORM]
Name: diagnoser, dtype: object

In [13]:
df["antall"] = df["diagnoser"].apply(len)
df["antall"].value_counts()

antall
1    14996
2     4229
3     1512
4      528
0      411
5      119
6        4
Name: count, dtype: int64

In [14]:
en_diagnose = df[df["antall"] == 1]
print(len(en_diagnose))


14996


In [15]:
en_diagnose["kode"] = en_diagnose["diagnoser"].str[0]
en_diagnose["kode"].value_counts()


kode
NORM       9069
NDT        1254
IMI        1024
LVH         456
NST_        435
LAFB        431
ASMI        409
CLBBB       353
IRBBB       311
ILMI        175
ISCAL       131
CRBBB       106
IVCD         98
AMI          91
WPW          71
ISCIN        67
1AVB         53
ISCIL        49
ALMI         49
LNGQT        46
LMI          37
ISCAS        36
ISC_         36
IPLMI        30
LAO/LAE      29
EL           26
INJAS        23
RAO/RAE      18
ISCLA        15
IPMI         15
DIG          14
INJAL        11
LPFB          6
PMI           5
SEHYP         4
ILBBB         3
RVH           2
INJIN         2
ISCAN         2
2AVB          2
3AVB          1
INJIL         1
Name: count, dtype: int64

In [16]:
en_diagnose["klasse"] = en_diagnose["kode"].map(diagnoser["diagnostic_subclass"])
en_diagnose["klasse"].value_counts()

klasse
NORM         9069
STTC         1340
IMI          1247
AMI           583
LVH           456
LAFB/LPFB     437
NST_          435
CLBBB         353
IRBBB         311
ISCA          184
ISCI          116
CRBBB         106
IVCD           98
WPW            71
_AVB           56
LMI            37
ISC_           36
LAO/LAE        29
RAO/RAE        18
PMI             5
SEHYP           4
ILBBB           3
RVH             2
Name: count, dtype: int64

In [17]:
def hent_sikkerhet(rad):
    return rad["scp_codes"][rad["kode"]]

en_diagnose["sikkerhet"] = en_diagnose.apply(hent_sikkerhet, axis=1)
en_diagnose["sikkerhet"].value_counts()

sikkerhet
100.0    11227
80.0      1772
50.0      1032
15.0       741
35.0       211
0.0         13
Name: count, dtype: int64

In [18]:
kandidater = en_diagnose[en_diagnose["sikkerhet"] >= 50]
kandidater = kandidater.sample(frac=1, random_state=42)
kandidater = kandidater.sort_values("sikkerhet", ascending=False, kind="stable")
print(len(kandidater))

14031


In [19]:
kandidater = kandidater.drop_duplicates("patient_id")
utvalg = kandidater.groupby("klasse").head(100)
utvalg["klasse"].value_counts()

klasse
NORM         100
NST_         100
IRBBB        100
IMI          100
STTC         100
CLBBB        100
AMI          100
LVH          100
ISCA         100
LAFB/LPFB    100
ISCI         100
CRBBB         98
IVCD          94
_AVB          52
WPW           49
ISC_          35
LAO/LAE       24
RAO/RAE       17
LMI           14
SEHYP          4
ILBBB          3
PMI            3
RVH            1
Name: count, dtype: int64

In [20]:
per_klasse = utvalg["klasse"].value_counts()
store_klasser = per_klasse[per_klasse >= 90].index
utvalg = utvalg[utvalg["klasse"].isin(store_klasser)]
print(len(utvalg))

1292


In [21]:
kolonner = ["patient_id", "klasse", "kode", "sikkerhet", "strat_fold", "filename_hr"]
utvalg[kolonner].sort_index().to_csv("utvalg.csv")
pd.read_csv("utvalg.csv").head()

,ecg_id,patient_id,klasse,kode,sikkerhet,strat_fold,filename_hr
0,16,10999.0,NORM,NORM,100.0,6,records500/00000/00016_hr
1,30,19053.0,LVH,LVH,100.0,9,records500/00000/00030_hr
2,41,8420.0,LAFB/LPFB,LAFB,100.0,9,records500/00000/00041_hr
3,49,13958.0,IVCD,IVCD,100.0,9,records500/00000/00049_hr
4,96,21698.0,LVH,LVH,100.0,1,records500/00000/00096_hr
